# Provenance for fault equivalence checks

Fault equivalence checks support returning the _reason_ that a check failed (if it does). In paritea, this is called _provenance_. To see how we can use this in action, let us set up a $3\times3\times2$ surface code memory experiment:

In [1]:
from paritea.generate import surface_code_memory_experiment
from paritea.drawing import draw

distance = 3
d, partitions = surface_code_memory_experiment(distance=distance, rounds=2, partition=True)
draw(d)

We can check that this implementation of memory with the surface code is fault tolerant up to its distance by creating two noise models and checking their fault equivalence:
1. A noise model `nm1` where all edges of the diagram are idealised
2. A noise model `nm2` where all edges of the diagram _but the ones connecting the two rounds_ are idealised

Here, we will only check CSS noise, so we set the weights of noise to $(1,2,1)$, effectively erasing any special weighting that $Y$ faults would introduce (on each edge, we could instead also just take the combination of $XZ$ and achieve the same lowest weight). Furthermore, we only need to set these parameters for the noise model that will actually contain faults:

In [2]:
from paritea.noise import NoiseModel

[first_part, second_part] = partitions

def _is_crossing_edge(e: int) -> bool:
    s, t = d.get_edge_endpoints_by_index(e)

    return (s in first_part and t in second_part) or (s in second_part and t in first_part)

nm1 = NoiseModel.weighted_edge_flip_noise(d, idealised_edges=d.edge_indices())
nm2 = NoiseModel.weighted_edge_flip_noise(
    d, w_x=1, w_y=2, w_z=1, idealised_edges=[e for e in d.edge_indices() if not _is_crossing_edge(e)]
)

Finally, we quickly check that the noise models are fault equivalent to exactly the distance of the surface code:

In [3]:
from paritea.equivalence.fault_equivalence import is_fault_equivalence, check_fault_equivalence

# The fault equivalence is valid until it fails at exactly weight == distance
violation = check_fault_equivalence(nm1, nm2)
assert violation.weight == distance

But what if we want to explore _why_ the check returned a violation? For this, we can enable the `provenance` mode via its corresponding keyword argument, which populates the field `violation.faults`:

In [4]:
violation = check_fault_equivalence(nm1, nm2, provenance=True)
print(violation.faults)

[(Fault(edge_flips=PauliString({117: PauliX}), detector_flips=frozenset()), 1), (Fault(edge_flips=PauliString({123: PauliZ}), detector_flips=frozenset()), 1), (Fault(edge_flips=PauliString({159: PauliY}), detector_flips=frozenset()), 2)]


These faults are not really readable as is, but since they are just Pauli strings, we can reduce them to a single string that we can display using the `draw` function:

In [5]:
from paritea.pauli import PauliString

combination = PauliString()
for (f, _) in violation.faults:
    combination *= f.edge_flips

print(f"The combined weight of the violation is {violation.weight}")
draw(d, web=combination)

The combined weight of the violation is 3
